# Whole U.S. — Matched Records Export

**Goal:** Produce a clean dataset of CoStar properties that successfully matched to the U.S. Census geocoder.  
**Input:** `data/1_derived/4_whole_zipcode_matcher_001-007.csv`  
**Output:** `data/1_derived/5_whole_matched_final.csv`  
**Filter:** `match == 'Match'` (970,482 rows out of 1,051,219 total)

In [1]:
import os
import glob
import pandas as pd

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.max_rows', 20)

In [2]:
# --- Load all matcher output files ---
IN_DIR = os.path.join('..', '..', 'data', '1_derived')
files = sorted(glob.glob(os.path.join(IN_DIR, '4_whole_zipcode_matcher_0*.csv')))

print(f'Reading {len(files)} files...')
df = pd.concat([pd.read_csv(f, low_memory=False) for f in files], ignore_index=True)
print(f'Total rows loaded: {len(df):,}')
print(df['match'].value_counts())

Reading 7 files...
Total rows loaded: 1,051,219
match
Match       970482
No_Match     73355
Tie           6697
Name: count, dtype: int64


In [3]:
# --- Filter to successfully matched records only ---
matched = df[df['match'] == 'Match'].copy()
print(f'Matched rows: {len(matched):,}')

Matched rows: 970,482


In [4]:
# --- Select CoStar address fields + Census geocoding results ---
KEEP_COLS = [
    # CoStar identifiers
    'LeaseCompId',
    'PropertyId',
    'PropertyName',
    'Market',
    'Submarket',
    # CoStar address fields
    'Address.DeliveryAddress',
    'Suite',
    'Address.Locality',
    'Address.County',
    'Address.PostalCode',
    'Address.RegionName',
    'Address.Subdivision',
    'Address.Country',
    # Census geocoding results
    'matched_address',
    'latitude',
    'longitude',
    'state_fips',
    'county_fips',
    'census_tract',
    'census_block',
    'geoid',
    # Match metadata
    'match',
    'zip_match_detail',
]

# Keep only columns that exist in the dataframe
cols = [c for c in KEEP_COLS if c in matched.columns]
out = matched[cols].reset_index(drop=True)
print(f'Output shape: {out.shape}')
out.head(3)

Output shape: (970482, 23)


,LeaseCompId,PropertyId,PropertyName,Market,Submarket,Address.DeliveryAddress,Suite,Address.Locality,Address.County,Address.PostalCode,...,matched_address,latitude,longitude,state_fips,county_fips,census_tract,census_block,geoid,match,zip_match_detail
0,113952192.0,445018.0,The Palisades Medical Office,Atlanta,Upper Buckhead,3200 Downwood Cir NW,350,Atlanta,Fulton,30327.0,...,"3200 DOWNWOOD CIR NW, ATLANTA, GA, 30327",33.840893,-84.426522,13.0,121.0,9803.0,2002.0,1.312101e+14,Match,One-to-One Matched
1,114008333.0,440921.0,100 Edgewood,Atlanta,Downtown Atlanta,100 Edgewood Ave NE,540,Atlanta,Fulton,30303.0,...,"100 EDGEWOOD AVE NE, ATLANTA, GA, 30303",33.754541,-84.386374,13.0,121.0,11901.0,3012.0,1.312101e+14,Match,One-to-One Matched
2,113897878.0,442552.0,Buckhead West - Bldg 1888,Atlanta,Lower Buckhead,1888 Emery St NW,NaN,Atlanta,Fulton,30318.0,...,"1888 EMERY ST NW, ATLANTA, GA, 30318",33.807229,-84.414403,13.0,121.0,9001.0,2001.0,1.312101e+14,Match,One-to-One Matched


In [5]:
# --- Write output ---
OUT_PATH = os.path.join(IN_DIR, '5_whole_matched_final.csv')
out.to_csv(OUT_PATH, index=False)
print(f'Saved {len(out):,} rows to {OUT_PATH}')

Saved 970,482 rows to ..\..\data\1_derived\5_whole_matched_final.csv
